In [65]:
import pandas as pd
import os
import numpy as np
import re

In [66]:
pd.set_option('display.max_columns', None)

# Data curation

## **STEP 1**. Merge dicomtocsv_series.csv and dicomtocsv_study.csv

### <span style="color:blue">**Main**</span>

In [67]:
file_path = "P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/R3Data"
# file_series_p1 = "dicomtocsv_series_20250527.xlsx"
file_series_p2 = "dicomtocsv_series_20260803.xlsx"
file_series_p3 = "dicomtocsv_series_20260812.xlsx"

In [68]:
# df_dicom_series_p1 = pd.read_excel(os.path.join(file_path, file_series_p1))
df_dicom_series_p2 = pd.read_excel(os.path.join(file_path, file_series_p2))
df_dicom_series_p3 = pd.read_excel(os.path.join(file_path, file_series_p3))

In [69]:
# df_dicom_series_p1.shape, df_dicom_series_p2.shape, df_dicom_series_p3.shape

In [70]:
# df_tmp = pd.concat((df_dicom_series_p1, df_dicom_series_p2), axis=0)
# df_all = pd.concat((df_tmp, df_dicom_series_p3), axis=0)
df_all = pd.concat((df_dicom_series_p2, df_dicom_series_p3), axis=0)

df_all.shape

(3136, 7)

In [71]:
df_all['StudyDate'] = pd.to_datetime(df_all['StudyDate'], format='%Y%m%d').dt.strftime('%Y-%m-%d')

In [72]:
df_all.head(5)

,PatientID,StudyDate,StudyDescription,AccessionNumber,SeriesNumber,SeriesDescription,FolderPath
0,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73200000,L CC Breast Tomosynthesis Image,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
1,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73200000,L MLO Breast Tomosynthesis Image,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
2,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73200000,R CC Breast Tomosynthesis Image,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
3,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73200000,R MLO Breast Tomosynthesis Image,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
4,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73500000,L CC Breast Tomosynthesis Image Slabs,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...


## **STEP 2**. Group by PatientID

### <span style="color:blue">**Main**</span>

In [73]:
df_sort = df_all.sort_values(
        by=['PatientID', 'StudyDate', 'AccessionNumber'],
        ignore_index=True
    )

num_patient = df_sort["PatientID"].nunique()
print(num_patient)

536


In [74]:
df_step2 = df_sort

In [75]:
df_step2[df_step2['PatientID'] == 4333000921]

,PatientID,StudyDate,StudyDescription,AccessionNumber,SeriesNumber,SeriesDescription,FolderPath
8,4333000921,2022-08-26,SCREENING MAMMOGRAM WITH TOMO,451046228,73500000,L CC Breast Tomosynthesis Image Slabs,J:\Testing\jlee\MO_DBT\376GB_Tomos_2nd_600\Pat...
9,4333000921,2022-08-26,SCREENING MAMMOGRAM WITH TOMO,451046228,73500000,L MLO Breast Tomosynthesis Image Slabs,J:\Testing\jlee\MO_DBT\376GB_Tomos_2nd_600\Pat...
10,4333000921,2022-08-26,SCREENING MAMMOGRAM WITH TOMO,451046228,73500000,R CC Breast Tomosynthesis Image Slabs,J:\Testing\jlee\MO_DBT\376GB_Tomos_2nd_600\Pat...
11,4333000921,2022-08-26,SCREENING MAMMOGRAM WITH TOMO,451046228,73500000,R MLO Breast Tomosynthesis Image Slabs,J:\Testing\jlee\MO_DBT\376GB_Tomos_2nd_600\Pat...


## **STEP 3.** Add tags (Study, Side, Series)

In [76]:
dicom = df_step2

In [77]:
dicom.head(3)

,PatientID,StudyDate,StudyDescription,AccessionNumber,SeriesNumber,SeriesDescription,FolderPath
0,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73200000,L CC Breast Tomosynthesis Image,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
1,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73200000,L MLO Breast Tomosynthesis Image,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
2,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73200000,R CC Breast Tomosynthesis Image,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...


In [78]:
dicom_copy = dicom.copy()

### <span style="color:blue"> **Study**</span> (SCREEN, DIAG)

In [79]:
study_types = {
    "DIAG":   ["DIAG", "DIAGNOSTIC", "DX"],
    "SCREEN": ["SCREENING", "SCREEN"],
}

In [80]:
column_to_check = 'StudyDescription'
type_column = 'Study'

dicom_copy[column_to_check] = dicom_copy[column_to_check].astype(str)

for label, terms in study_types.items():
    pattern = '|'.join(re.escape(t) for t in terms)
    mask = dicom_copy[column_to_check].str.contains(pattern, case=False, na=False, regex=True)
    dicom_copy.loc[mask, type_column] = label

### <span style="color:blue">**Side**</span> (R, L)

In [81]:
side_types = {
    "R": ["RIGHT", "RT", "R XCCL", "R MLO", "R CC", "R ML", "R SIO", "R LM"],
    "L": ["LEFT", "LT", "L XCCL", "L MLO", "L CC", "L ML", "L SIO", "L LM"],
}

In [82]:
column_to_check = 'SeriesDescription'
type_column = 'Side'

dicom_copy[column_to_check] = dicom_copy[column_to_check].astype(str)

for label, terms in side_types.items():
    pattern = '|'.join(re.escape(t) for t in terms)
    mask = dicom_copy[column_to_check].str.contains(pattern, case=False, na=False, regex=True)
    dicom_copy.loc[mask, type_column] = label

### <span style="color:blue">**Series**</span> (DBT, IN2D, C VIEW, SECURE)

In [83]:
series_types = {
    "DBT":    ["Breast Tomosynthesis"],
    "IN2D":   ["Intelligent 2D"],
    "C VIEW": ["C-View"],
    "SECURE": ["SecurView", "CAD SC"],
}

ffdm_exact = ["R CC", "R MLO", "R ML", "L CC", "L MLO", "L ML",
              "L XCCL", "R XCCL", "R LM", "L LM",
              "R SIO", "L SIO", "RT", "LT"
              ]

In [84]:
column_to_check = 'SeriesDescription'
type_column = 'Series'

dicom_copy[column_to_check] = dicom_copy[column_to_check].astype(str)

for label, terms in series_types.items():
    pattern = '|'.join(re.escape(t) for t in terms)
    mask = dicom_copy[column_to_check].str.contains(pattern, case=False, na=False, regex=True)
    dicom_copy.loc[mask, type_column] = label

# Exact matches for FFDM
ffdm_mask = dicom_copy[column_to_check].str.strip().str.upper().isin(
    [t.upper() for t in ffdm_exact]
)
dicom_copy.loc[ffdm_mask, type_column] = "FFDM"

### <span style="color:blue">**View**</span> (MLO, CC)

In [85]:
view_types = {
    "MLO": ["L MLO", "R MLO"],
    "CC":  ["L CC", "R CC"],
    "ML":  ["L ML", "R ML"],
    "XCCL":  ["L XCCL", "R XCCL"],
    "LM":  ["L LM", "R LM"],
    "SIO": ["R SIO", "L SIO"],
}

In [86]:
column_to_check = 'SeriesDescription'
type_column = 'View'

dicom_copy[column_to_check] = dicom_copy[column_to_check].astype(str)

for label, terms in view_types.items():
    pattern = '|'.join(re.escape(t) for t in terms)
    mask = dicom_copy[column_to_check].str.contains(pattern, case=False, na=False, regex=True)
    dicom_copy.loc[mask, type_column] = label

### <span style="color:blue">**Reorder columns**</span>

In [87]:
dicom_copy.columns

Index(['PatientID', 'StudyDate', 'StudyDescription', 'AccessionNumber',
       'SeriesNumber', 'SeriesDescription', 'FolderPath', 'Study', 'Side',
       'Series', 'View'],
      dtype='object')

In [88]:
dicom_copy = dicom_copy[['PatientID', 
        'AccessionNumber', 'Study', 'Side', 'Series', 'View',    
        'StudyDate', 'StudyDescription',
        'SeriesDescription', 'SeriesNumber', 
        'FolderPath'
        ]]

In [89]:
dicom_copy.head(3)

,PatientID,AccessionNumber,Study,Side,Series,View,StudyDate,StudyDescription,SeriesDescription,SeriesNumber,FolderPath
0,4333000414,455560367,SCREEN,L,DBT,CC,2021-06-12,SCREENING MAMMO WITH TOMO,L CC Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
1,4333000414,455560367,SCREEN,L,DBT,ML,2021-06-12,SCREENING MAMMO WITH TOMO,L MLO Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
2,4333000414,455560367,SCREEN,R,DBT,CC,2021-06-12,SCREENING MAMMO WITH TOMO,R CC Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...


### <span style="color:#FF6347;">**SAVE**</span> file

In [90]:
path = "P:/Dataset/R01-MO-DBT/MO-DBT-data-curation"

In [91]:
output_file = os.path.join(path,'dicom_tag_p2' + ".xlsx")
dicom_copy.to_excel(output_file, index=False)

### <span style="color:#FF6347;">**READ**</span> file

In [92]:
file_path = os.path.join(path,'dicom_tag_p2' + ".xlsx")
dicom = pd.read_excel(file_path)

In [93]:
dicom[dicom["Series"]=="DBT"]

,PatientID,AccessionNumber,Study,Side,Series,View,StudyDate,StudyDescription,SeriesDescription,SeriesNumber,FolderPath
0,4333000414,455560367,SCREEN,L,DBT,CC,2021-06-12,SCREENING MAMMO WITH TOMO,L CC Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
1,4333000414,455560367,SCREEN,L,DBT,ML,2021-06-12,SCREENING MAMMO WITH TOMO,L MLO Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
2,4333000414,455560367,SCREEN,R,DBT,CC,2021-06-12,SCREENING MAMMO WITH TOMO,R CC Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
3,4333000414,455560367,SCREEN,R,DBT,ML,2021-06-12,SCREENING MAMMO WITH TOMO,R MLO Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
4,4333000414,455560367,SCREEN,L,DBT,CC,2021-06-12,SCREENING MAMMO WITH TOMO,L CC Breast Tomosynthesis Image Slabs,73500000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
...,...,...,...,...,...,...,...,...,...,...,...
3131,4333669308,453037184,SCREEN,R,DBT,ML,2022-06-07,SCREENING MAMMOGRAM WITH TOMO,R MLO Breast Tomosynthesis Image Slabs,73500000,J:\Testing\jlee\MO_DBT\376GB_Tomos_2nd_600\Pat...
3132,4333669687,454228662,SCREEN,L,DBT,CC,2021-10-13,SCREEN MAMMO WITH TOMO BILATERAL,L CC Breast Tomosynthesis Image Slabs,73500000,J:\Testing\jlee\MO_DBT\376GB_Tomos_2nd_600\Pat...
3133,4333669687,454228662,SCREEN,L,DBT,ML,2021-10-13,SCREEN MAMMO WITH TOMO BILATERAL,L MLO Breast Tomosynthesis Image Slabs,73500000,J:\Testing\jlee\MO_DBT\376GB_Tomos_2nd_600\Pat...
3134,4333669687,454228662,SCREEN,R,DBT,CC,2021-10-13,SCREEN MAMMO WITH TOMO BILATERAL,R CC Breast Tomosynthesis Image Slabs,73500000,J:\Testing\jlee\MO_DBT\376GB_Tomos_2nd_600\Pat...


In [94]:
dicom[dicom["Series"]=="DBT"]["PatientID"].unique().size

536

In [95]:
dicom[dicom['PatientID']==4333000414]

,PatientID,AccessionNumber,Study,Side,Series,View,StudyDate,StudyDescription,SeriesDescription,SeriesNumber,FolderPath
0,4333000414,455560367,SCREEN,L,DBT,CC,2021-06-12,SCREENING MAMMO WITH TOMO,L CC Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
1,4333000414,455560367,SCREEN,L,DBT,ML,2021-06-12,SCREENING MAMMO WITH TOMO,L MLO Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
2,4333000414,455560367,SCREEN,R,DBT,CC,2021-06-12,SCREENING MAMMO WITH TOMO,R CC Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
3,4333000414,455560367,SCREEN,R,DBT,ML,2021-06-12,SCREENING MAMMO WITH TOMO,R MLO Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
4,4333000414,455560367,SCREEN,L,DBT,CC,2021-06-12,SCREENING MAMMO WITH TOMO,L CC Breast Tomosynthesis Image Slabs,73500000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
5,4333000414,455560367,SCREEN,L,DBT,ML,2021-06-12,SCREENING MAMMO WITH TOMO,L MLO Breast Tomosynthesis Image Slabs,73500000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
6,4333000414,455560367,SCREEN,R,DBT,CC,2021-06-12,SCREENING MAMMO WITH TOMO,R CC Breast Tomosynthesis Image Slabs,73500000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
7,4333000414,455560367,SCREEN,R,DBT,ML,2021-06-12,SCREENING MAMMO WITH TOMO,R MLO Breast Tomosynthesis Image Slabs,73500000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...


In [96]:
dicom[dicom['PatientID']==4333004686]

,PatientID,AccessionNumber,Study,Side,Series,View,StudyDate,StudyDescription,SeriesDescription,SeriesNumber,FolderPath
70,4333004686,455580593,DIAG,L,DBT,CC,2021-07-06,DIAGNOSTIC LEFT UNI MAMMO WITH TOMO,L CC Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
71,4333004686,455580593,DIAG,L,DBT,ML,2021-07-06,DIAGNOSTIC LEFT UNI MAMMO WITH TOMO,L MLO Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
72,4333004686,455580593,DIAG,R,DBT,CC,2021-07-06,DIAGNOSTIC LEFT UNI MAMMO WITH TOMO,R CC Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
73,4333004686,455580593,DIAG,R,DBT,ML,2021-07-06,DIAGNOSTIC LEFT UNI MAMMO WITH TOMO,R MLO Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
74,4333004686,455580593,DIAG,L,DBT,CC,2021-07-06,DIAGNOSTIC LEFT UNI MAMMO WITH TOMO,L CC Breast Tomosynthesis Image Slabs,73500000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
75,4333004686,455580593,DIAG,L,DBT,ML,2021-07-06,DIAGNOSTIC LEFT UNI MAMMO WITH TOMO,L MLO Breast Tomosynthesis Image Slabs,73500000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
76,4333004686,455580593,DIAG,R,DBT,CC,2021-07-06,DIAGNOSTIC LEFT UNI MAMMO WITH TOMO,R CC Breast Tomosynthesis Image Slabs,73500000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
77,4333004686,455580593,DIAG,R,DBT,ML,2021-07-06,DIAGNOSTIC LEFT UNI MAMMO WITH TOMO,R MLO Breast Tomosynthesis Image Slabs,73500000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
78,4333004686,451323312,SCREEN,L,DBT,CC,2022-07-18,SCREENING MAMMO WITH TOMO,L CC Breast Tomosynthesis Image Slabs,73500000,J:\Testing\jlee\MO_DBT\376GB_Tomos_2nd_600\Pat...
79,4333004686,451323312,SCREEN,L,DBT,ML,2022-07-18,SCREENING MAMMO WITH TOMO,L MLO Breast Tomosynthesis Image Slabs,73500000,J:\Testing\jlee\MO_DBT\376GB_Tomos_2nd_600\Pat...
